# 52 Timflow model: calibration, analysis and storm event

<!-- navigation -->
[← 51 Prepare model inputs](51%20Prepare%20model%20inputs.ipynb) | [Overview](README.md) | [60 Summary over transects →](60%20Summary%20over%20transects.ipynb)

| | |
|---|---|
| **Workflow** | minimal path, step 5 (the storm event is optional) |
| **Input** | `observations_model_period.csv` (notebook 51), `model_parameters_<transect>.xlsx`, transects metadata, piezometers, storm event file (optional) |
| **Output** | calibrated parameters (new tab in the parameter file), figures and response factors in the transect result folders |

Builds the steady and transient Timflow models of one transect, calibrates them on the observations, analyses the calibrated model (leakage factors, response factors) and computes the heads during a storm event.

In [ ]:
from pathlib import Path

import timflow_levee_toolkit as tlt

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import timflow as tf

print(f"timflow_levee_toolkit {tlt.__version__}")
tf.show_versions()

In [ ]:
## UPDATE FOR YOUR PROJECT: the project folder (with project_settings.toml and the data)
project_dir = Path.cwd().parent / "examples" / "papendrecht"

project = tlt.load_project(project_dir)
units = project.units
project

In [ ]:
## UPDATE FOR YOUR PROJECT: the transect to work on (a name from the transects metadata)
transect_name = "Papendrecht West"

## 1. Inputs
Metadata, piezometers and the observations of the model period (from notebook 51).

In [ ]:
location_name = f"{project.name} - {transect_name}"

meta = project.transect_metadata(transect_name)
river_loc1 = meta["loc_river"]
loc_storm_event = meta["loc_storm_event"]
loc_farfield = meta["farfield"]
L_river = meta["L_river"]
L_foreland = meta["L_foreland"]
L_hinterland = meta["L_hinterland"]  # only for plots: in the model the hinterland extends to infinity

obslocs = tlt.make_obslocs(project.piezometers(transect_name))

obs_ref = pd.read_csv(project.transect_dir(transect_name, "timeseries") / "observations_model_period.csv", index_col=0)
obs_dh = obs_ref - obs_ref.mean()

savefolder = project.transect_dir(transect_name, "intermediate")
savefolder_final = project.transect_dir(transect_name, "final")

obs_summary = tlt.summarize_observations(obs_ref, obslocs, loc_farfield, river_loc1)
obs_avg_farfieldref = obs_summary["h_avg_farfield"]

Calculate the `tsandhstar` boundary condition at the river for the daily (**tidal**) situation: tidal fluctuation at the river minus the average river level.

_See the [Timflow documentation](https://timflow.readthedocs.io/en/latest/transient/00userguide/howtos/howto_fluctuating_head_boundary.html) for why the midpoint of a time interval is used. This only influences the time scale, not the amplitudes._

In [ ]:
tsandhstar_tidal = tlt.to_tsandhstar(obs_dh[river_loc1])
print(f"Tidal situation: {len(tsandhstar_tidal)} steps")

## 2. Geometry and initial parameters
Set initial parameters based on geological data. Use consistent units for length `[m]` and time `[days]`.

### 2.1 Parameters from the Excel file

In [ ]:
excel_tab  = 'initial'   # Tab name: change to 'calibrated' etc. when needed
excel_path = project.parameters_file(transect_name)

inhom_params, naq = tlt.read_model_params(excel_path, tab_name=excel_tab)

tlt.print_model_params(inhom_params)

### 2.2 Create the models (steady state + transient)
The models `mlss` and `mltr` are created below, for the steady and the transient behaviour. `naq` is the number of aquifers.

<div class="alert alert-info">
    <strong>Note:</strong> In this example the river extends to -infinity. If the river has a finite width, you can model a water divide in the center of the river. This can be accomplished in several ways:
    <ol>
        <li> by adding an impermeable wall (water divide) in the middle of the river</li>
        <li> by adding an extra inhomogeneity ranging from -infinity to the left embankment of the river</li>
    </ol>
    Method 1 is shown below.
</div>

_The models are created explicitly here. Once Timflow can save and load models ([PR #171](https://github.com/timflow-org/timflow/pull/171)), this notebook can be split into separate notebooks._

In [ ]:
# NOTE: The models are created explicitly in this notebook. Once Timflow can save and load models
# (JSON, see https://github.com/timflow-org/timflow/pull/171), this notebook can be split into
# separate notebooks for calibration, analysis and the storm event, that load the calibrated model.

hstar_river = obs_ref[river_loc1].mean()    # Average head in river
hstar_polder = obs_ref[loc_farfield].mean() # Farfield reference

mlss = tf.steady.ModelXsection(naq=naq)

riv_ss = tf.steady.XsectionMaq(
    mlss,
    x1=-np.inf,  # river extends to -infinity
    x2=-L_foreland,
    kaq=inhom_params["river"]["kaq"],
    z=inhom_params["river"]["z"],
    c=inhom_params["river"]["c"],
    topboundary="semi",
    hstar=hstar_river,
    name="river",
)
fore_ss = tf.steady.XsectionMaq(
    mlss,
    x1=-L_foreland,
    x2=0,
    kaq=inhom_params["foreland"]["kaq"],
    z=inhom_params["foreland"]["z"],
    c=inhom_params["foreland"]["c"],
    topboundary="semi",
    hstar=hstar_river,  # This may not be correct for daily situation
    name="foreland",
)
hinter_ss = tf.steady.XsectionMaq(
    mlss,
    x1=0,
    x2=np.inf,
    kaq=inhom_params["hinterland"]["kaq"],
    z=inhom_params["hinterland"]["z"],
    c=inhom_params["hinterland"]["c"],
    topboundary="semi",
    hstar=hstar_polder,
    name="hinterland",
)

x_rivercenter = - L_foreland - L_river / 2
boundary_noflow_ss = tf.steady.LeakyWall1D(
    mlss, x_rivercenter, res=np.inf, layers=list(range(naq)), label="no flow boundary"
)

mlss.solve()

Below, the transient model is defined. We deliberately do not add the steady-state model to the transient model when creating it, because we want to calculate response factors separately for the steady and the transient behaviour. The calibration takes both the steady and the transient part into account.

In [ ]:
mltr = tf.transient.ModelXsection(naq=naq, tmin=1e-5, tmax=1e2)

riv_tr = tf.transient.XsectionMaq(
    mltr,
    x1=-np.inf,  # river extends to -infinity
    x2= -L_foreland,
    kaq=inhom_params["river"]["kaq"],
    z=inhom_params["river"]["z"],
    Saq=inhom_params["river"]["Saq"],
    Sll=inhom_params["river"]["Sll"],
    c=inhom_params["river"]["c"],
    leffaq=inhom_params["river"]["leffaq"],
    leffll=inhom_params["river"]["leffll"],
    topboundary="semi",
    tsandhstar=tsandhstar_tidal,
    name="river",
)
fore_tr = tf.transient.XsectionMaq(
    mltr,
    x1=-L_foreland,
    x2=0,
    kaq=inhom_params["foreland"]["kaq"],
    z=inhom_params["foreland"]["z"],
    Saq=inhom_params["foreland"]["Saq"],
    Sll=inhom_params["foreland"]["Sll"],
    c=inhom_params["foreland"]["c"],
    leffaq=inhom_params["foreland"]["leffaq"],
    leffll=inhom_params["foreland"]["leffll"],
    topboundary="semi",
    tsandhstar=None,  # In daily situation, the foreland is dry
    name="foreland",
)
hinter_tr = tf.transient.XsectionMaq(
    mltr,
    x1=0,
    x2=np.inf,
    kaq=inhom_params["hinterland"]["kaq"],
    z=inhom_params["hinterland"]["z"],
    Saq=inhom_params["hinterland"]["Saq"],
    Sll=inhom_params["hinterland"]["Sll"],
    c=inhom_params["hinterland"]["c"],
    leffaq=inhom_params["hinterland"]["leffaq"],
    leffll=inhom_params["hinterland"]["leffll"],
    topboundary="semi",
    name="hinterland",
)

x_rivercenter = -L_foreland - L_river / 2
boundary_noflow_tr = tf.transient.LeakyWall1D(mltr, x_rivercenter, res=np.inf, layers=list(range(naq)), label="no flow boundary")

mltr.solve()

### 2.3 Calibration object

In [ ]:
cal = tf.Calibrate(steady_model=mlss, transient_model=mltr)

# Select only obslocs that do not have a value 'use_calibration'= False in the Geopackage
if "use_calibration" in obslocs.columns:
    obslocs_calib = obslocs[obslocs["use_calibration"] != 0]
else:
    obslocs_calib = obslocs


# Add all locations for which 'use_calibration' is not 0
for pb, row in obslocs_calib.iterrows():
    if pb not in obs_ref.columns:
        print(f"location {pb} skipped: no observations in the selected period")
        continue

    hobs = obs_ref[pb].dropna()
    cal.add_head_time_series(
        pb,
        x=row["x_crosssection"],
        y=0,
        layer=int(row["modellayer"]),
        t=hobs.index.to_numpy(),
        h=hobs.values,
    )
    print(f"location {pb} added to calibration")

### 2.4 Cross-section of the geohydrological model
- model layout with layer numbers
- model layout with initial parameters

In [ ]:
# Some plotting parameters for all xsection plots
xsec_xlim = (-L_foreland - L_river / 2 - 10, 250)
xsec_zlim = (None, 8)
xsec_figsize = (10, 7)

In [ ]:
# Plot the model layout
f, ax = tlt.plot_xsection(
    mltr,
    model_steady=mlss,
    calibrate=cal,
    xlim=xsec_xlim,
    zlim=xsec_zlim,
    figsize=xsec_figsize,
    axis_units=units,
    labels=True,
    params=False,
)
tlt.finalize_and_save(
    f, savefolder, location_name, modelrun="Model layout", tag="xsection", file_label=transect_name
)

# Plot the model layout with parameters and inhomogeneity names
f, ax = tlt.plot_xsection(
    mltr,
    model_steady=mlss,
    calibrate=cal,
    xlim=xsec_xlim,
    zlim=xsec_zlim,
    figsize=xsec_figsize,
    axis_units=units,
)
tlt.finalize_and_save(
    f, savefolder, location_name, modelrun="Run 0 Initial parameters", tag="xsection", file_label=transect_name
)

## 3. Calibration
### 3.1 Tidal fluctuation before calibration
Modelled versus observed heads at the observation wells.

In [ ]:
modelrun = "Run 1 tidal"

f, axes = tlt.plot_transient_results(cal, units=units)
tlt.finalize_and_save(f, savefolder, location_name, modelrun, tag="headT", file_label=transect_name)

### 3.2 Calibration

Choose the parameters to calibrate. The example below is the set used for the Papendrecht transects, for a scheme
with two aquifers: the resistances of both aquitards (below the river, and below foreland and hinterland together),
the conductivity of the upper aquifer, the storage of the deep aquifer and its loading efficiency below river and
foreland. Fewer parameters gave a clearly worse fit there (RMSE about 0.16 m instead of 0.10 m), mainly because the
shallow piezometers need the top aquitard and the upper aquifer. Adapt the set to your scheme: look at the fit and
at the correlations in the report, and add or remove parameters one at a time.

`cal.set_aquifer_parameter(name, layers, initial, pmin, pmax, log_scale, inhoms, model)`:
- `name`: `kaq`, `c`, `Saq`, `Sll` or `kzoverkh`
- `layers`: the layer(s); consecutive layers get one value. `c` of layer *i* is the resistance of the aquitard above aquifer *i*.
- `inhoms`: the inhomogeneities that share one value (`river`, `foreland`, `hinterland`)
- `initial`: start value; left out = the value in the geohydrological scheme
- `log_scale=True` is recommended for `kaq`, `c`, `Saq` and `Sll`
- `model`: `both` (default), `steady` or `transient` (storage only matters in the transient model)

The loading efficiency (`leffaq`, `leffll`) is not an aquifer parameter in Timflow: it is calibrated on the array of
the transient inhomogeneity. To share one value between river and foreland, the foreland is added as an extra target
(an internal function, see [Timflow issue #186](https://github.com/timflow-org/timflow/issues/186)).

In [ ]:
# Calibration parameters of the Papendrecht transects (two aquifers); start values from the geohydrological scheme
cal.set_aquifer_parameter("c", layers=0, pmin=0.01, pmax=50, log_scale=True, inhoms=["river"])
cal.set_aquifer_parameter("c", layers=0, pmin=50, pmax=1000, log_scale=True, inhoms=["foreland", "hinterland"])
cal.set_aquifer_parameter("kaq", layers=0, pmin=0.01, pmax=20, log_scale=True, inhoms=["foreland", "hinterland"])
cal.set_aquifer_parameter("c", layers=1, pmin=2, pmax=500, log_scale=True, inhoms=["river"])
cal.set_aquifer_parameter("c", layers=1, pmin=500, pmax=5000, log_scale=True, inhoms=["foreland", "hinterland"])
cal.set_aquifer_parameter(
    "Saq", layers=1, pmin=1e-6, pmax=1e-2, log_scale=True, inhoms=["river", "foreland", "hinterland"], model="transient"
)

# Loading efficiency of the deep aquifer, one value for river and foreland (see the note above)
cal.set_parameter_by_reference("leffaq", riv_tr.leffaq[1:2], initial=None, pmin=0.0, pmax=1.0)
cal._parameters["leffaq"].add_target(fore_tr.leffaq[1:2], slice(None))

In [ ]:
# Calibrate with Levenberg-Marquardt (lmfit), which also reports the correlations between the parameters
cal.lmfit()
rmse = cal.rmse()  # store now: later cells change the models
print(f"RMSE: {rmse:.5f}")

The RMSE (root mean square error) is a measure of the accuracy of the fit. An RMSE of 0.01 m means an average difference of 1 cm between modelled and measured hydraulic heads.

In [ ]:
cal.result

In [ ]:
modelrun = "Run 2 calibrated"

f, ax = tlt.plot_xsection(
    mltr,
    model_steady=mlss,
    calibrate=cal,
    xlim=xsec_xlim,
    zlim=xsec_zlim,
    figsize=xsec_figsize,
    axis_units=units,
)
tlt.finalize_and_save(
    f, savefolder, location_name, modelrun=modelrun, tag="xsection", file_label=transect_name
)

Plot modeled vs observed heads after calibration.

In [ ]:
f, axes = tlt.plot_transient_results(cal, units=units)
tlt.finalize_and_save(f, savefolder, location_name, modelrun=modelrun, tag="headT", file_label=transect_name)

Show parameters dataframe

In [ ]:
cal.parameters.iloc[:, :-2]

Check if parameters are correctly linked: `kaq`, `Saq` in all inhoms, `c`
separate in each section.

In [ ]:
mlss.aquifer_summary().style.format(na_rep="-", precision=2)

In [ ]:
(
    mltr.aquifer_summary()
    .style.format(na_rep="-", precision=2)
    .format("{:.2e}", subset=["S_s"])
)

## 4. Analysis of the calibrated model

### 4.1 Leakage factors
For the leakage factors we use the lambda values calculated by Timflow. They are sorted from large to small; the largest one (the first value) is the most important.

_Read more in [Timflow issue 187](https://github.com/timflow-org/timflow/issues/187)._

In [ ]:
leakagefactor_riverside = mlss.aq.inhomdict["river"].lab[0]
print(f"Leakagefactor riverside: {leakagefactor_riverside:5.0f} m")

leakagefactor_landside = mlss.aq.inhomdict["hinterland"].lab[0]
print(f"Leakagefactor landside:  {leakagefactor_landside:5.0f} m")

print(f"Ratio between leakagefactors: {leakagefactor_riverside / leakagefactor_landside:5.2f}")


flow_resistance_foreland = tlt.calculate_flow_resistance(
    mlss, "foreland", layer=-1
)
print(f"Flow resistance foreland W1: {flow_resistance_foreland:5.3f} d/m")

flow_resistance_hinterland = tlt.calculate_flow_resistance(
    mlss, "hinterland", layer=-1
)
print(f"Flow resistance hinterland W3: {flow_resistance_hinterland:5.3f} d/m")

print(
    f"Ratio W1/W3: {flow_resistance_foreland / flow_resistance_hinterland :5.2f}"
)

### 4.2 Tidal response factors

We use a transient model that does not contain the steady part. Therefore we can directly calculate tidal responsefactors.

In [ ]:
modelrun = "Run 2 calibrated"
t_tidal = obs_dh.index.to_numpy()
h_river_tidal = obs_dh[river_loc1]

# Heads relative to their average per piezometer: modeled (transient part only) and observed
h_mod_tidal = tlt.select_obslocs(
    tlt.calculate_head_t(mltr, obslocs["x_crosssection"].unique(), t_tidal), obslocs
).droplevel("layer", axis=1)
h_obs_tidal = obs_dh[[p for p in obslocs.index if p in obs_dh.columns]]

# Response factor and time lag per tidal peak, then the median per piezometer
peaks_tidal = {
    "mod": tlt.calculate_rf_peaks(h_mod_tidal, h_river_tidal),
    "obs": tlt.calculate_rf_peaks(h_obs_tidal, h_river_tidal),
}
df_rf_tidal = tlt.summarize_rf_tidal(peaks_tidal, obslocs)

df_rf_tidal

Response factor and time lag per tidal peak, for each piezometer (modeled and observed).

In [ ]:
for obsloc, row in obslocs.iterrows():
    f, axes = tlt.plot_rf_peaks(
        obsloc,
        h_river_tidal,
        series={"mod": h_mod_tidal[obsloc], "obs": h_obs_tidal.get(obsloc)},
        peaks=peaks_tidal,
        title=f"{obsloc} (x={row['x_crosssection']}, layer={int(row['modellayer'])})",
        units=units,
    )
    tlt.finalize_and_save(f, savefolder, transect_name, modelrun=modelrun, tag=f"peak_response_{obsloc}", file_label=transect_name)

### 4.3 Hysteresis
A small example of how hysteresis could be shown.

In [ ]:
obsloc = obslocs.index[0]
xloc = obslocs.loc[obsloc, "x_crosssection"]
modellayer = int(obslocs.loc[obsloc, "modellayer"])
t_hysteresis = obs_dh.index.to_numpy()
h_river_hysteresis = obs_dh[river_loc1]
h_mod = pd.Series(
    mltr.head(xloc, 0.0, t_hysteresis)[modellayer], index=obs_dh.index, name=f"{obsloc} (modeled)"
)

ax = tlt.plot_hysteresis(h_river_hysteresis, h_mod, units=units)
ax.set_title(f"{obsloc} (x={xloc}, layer={modellayer})", loc="right")

print(tlt.calculate_max_response(h_river_hysteresis, h_mod))

### 4.4 Steady state
The steady state is in the model `mlss`. Heads along x:

In [ ]:
modelrun = "Calibrated steady"

f, axes = tlt.plot_steady_results(cal,
                                  x1=- L_foreland - 5,
                                  x2=200,
                                  sharey=True,
                                  units=units
                                  )
tlt.finalize_and_save(f, savefolder, location_name, modelrun, tag="headX", file_label=transect_name)

### 4.5 Steady-state response factors

In [ ]:
df_rf_steady = tlt.calculate_rf_steady(
    mlss, obslocs, obs_avg_farfieldref[river_loc1], h_obs=obs_avg_farfieldref
)

df_rf_steady

### 4.6 Compare with the exponential decay function (steady state)

In [ ]:
f, ax = plt.subplots()
fit_steady = tlt.fit_response_profile(df_rf_steady, "rf_steady_mod", ax=ax, units=units)
ax.set_title("Response of the average head profile")
print(
    f"Leakage length riverside: {fit_steady['leakage_length_foreland']:.0f} m, "
    f"hinterland: {fit_steady['leakage_length_hinterland']:.0f} m, R² = {fit_steady['r_squared']:.3f}"
)

## 5. Save the calibrated parameters
If you are happy with the model, save the calibrated parameters as a new tab in the parameter file. If the tab name already exists, a timestamp is added. The tab can be read again with `read_model_params`, e.g. as the start of a next calibration.

_Saving and loading complete models (including the geometry) is being developed in Timflow, see [PR #171](https://github.com/timflow-org/timflow/pull/171)._

In [ ]:
mp_calibrated = tlt.write_model_params(mltr, excel_path, tab_name="calibrated")

In [ ]:
modelrun = "Run 50 accepted model"

# Plot the model layout with parameters and inhomogeneity names
f, ax = tlt.plot_xsection(
    mltr,
    model_steady=mlss,
    calibrate=cal,
    xlim=xsec_xlim,
    zlim=xsec_zlim,
    figsize=xsec_figsize,
    axis_units=units,
)
tlt.finalize_and_save(
    f, savefolder, location_name, modelrun=modelrun, tag="xsection", file_label=transect_name
)

## 6. Storm event ('waterstandsverloop')
Use the calibrated model to compute the groundwater heads during a storm event. This part is optional and needs a storm event file (`loc_storm_event` in the metadata).

### 6.1 Load the storm event
The river is assumed to be at its average level, so the changes compared to the average river level are used, starting at t = 0.

In [ ]:
fname_storm_event = project.storm_event_file(loc_storm_event)
df_storm_event_offset = tlt.read_storm_event(
    fname_storm_event, h_river_avg=obs_ref[river_loc1].mean()
)
f, axes = tlt.plot_storm_event(df_storm_event_offset, units=units)

tsandhstar_storm_event = tlt.to_tsandhstar(df_storm_event_offset["h_river_rel"])
print(f"storm event: {tsandhstar_storm_event}")

### 6.2 Assign new boundary condition

In the storm event, the foreland is wet. So here we assign the `tsandhstar` to the foreland as well.

In [ ]:
mltr.aq.inhomdict["river"].tsandhstar = tsandhstar_storm_event
mltr.aq.inhomdict["foreland"].tsandhstar = tsandhstar_storm_event
mltr.solve()

### 6.3 Plot head along T at multiple X locations

Plot of heads (versus time) at given x-locations. Here we use the locations of the observation points (`xlocs=obslocs["x_crosssection"].unique()`), but you could as well use specific locations of interest (`xlocs = np.array([0, 10, 100])`).

In [ ]:
# Select the x-locations to plot heads for:
xlocs = obslocs["x_crosssection"].unique()
# xlocs = np.array([0, 10, 100])

modelrun = "Run 51 storm event"

# Total head (steady + transient) [m NAP] over time, all layers
t = df_storm_event_offset.index.to_numpy()
h_t = tlt.calculate_head_t(mltr, xlocs, t, model_steady=mlss)

f, axes = tlt.plot_head_t(h_t, river=df_storm_event_offset["h_river"], units=units)
tlt.finalize_and_save(f, savefolder_final, location_name, modelrun=modelrun, tag="headTriv", file_label=transect_name)

#### 6.3.1 Response factor at the design high water (MHW)
MHW is the Dutch *maatgevend hoogwater*: the design high water level, here the peak of the storm event.

In [ ]:
modelrun = "Run 51 storm event"
h_river = df_storm_event_offset["h_river_rel"]

# Response factor over time: transient head (without steady part) / river head,
# per piezometer in its own model layer
h_tr_t = tlt.calculate_head_t(mltr, obslocs["x_crosssection"].unique(), h_river.index.to_numpy())
rf_t = tlt.select_obslocs(h_tr_t, obslocs).div(h_river, axis=0)
df_rf_mhw = tlt.calculate_rf_mhw(rf_t, obslocs, t_peak=h_river.idxmax())

f, axes = tlt.plot_head_t(rf_t, markers=df_rf_mhw[["t_mhw_mod", "rf_mhw_mod"]], ylabel="response factor [-]", units=units)
tlt.finalize_and_save(f, savefolder_final, location_name, modelrun=modelrun, tag="responsefactor_peak", file_label=transect_name)

df_rf_mhw

#### 6.3.2 Compare with exponential decay function

In [ ]:
f, ax = plt.subplots()
fit_mhw = tlt.fit_response_profile(df_rf_mhw, "rf_mhw_mod", ax=ax, units=units)
ax.set_title("Response of the head profile during MHW")
print(
    f"Leakage length riverside: {fit_mhw['leakage_length_foreland']:.0f} m, "
    f"hinterland: {fit_mhw['leakage_length_hinterland']:.0f} m, R² = {fit_mhw['r_squared']:.3f}"
)

### 6.4 Plot head along X at multiple times

The maximum of the presented time steps is calculated.

In [ ]:
t = np.arange(0.1, 6.5, 0.5)
x = np.linspace(-L_river - L_foreland, 300, 501)

# Heads along x: steady (layer, x) + transient (layer, time, x) = total [m NAP]
h_ss = mlss.headalongline(x, np.zeros_like(x))
h_tr = mltr.headalongline(x, np.zeros_like(x), t)
h_total = h_ss[:, np.newaxis] + h_tr

curves = {f"t = {ti:.1f} d": h_total[:, i] for i, ti in enumerate(t)}
curves["max"] = h_total.max(axis=1)
f, axes = tlt.plot_head_x(
    x, curves, styles={"max": {"color": "k", "linestyle": ":"}}, cmap="Spectral", units=units
)
tlt.finalize_and_save(f, savefolder_final, location_name, modelrun=modelrun, tag="headX2", file_label=transect_name)

## 7. Maximum heads during the storm in m NAP
The **maximum head during storm conditions in m NAP** is the maximum over time of the total head (steady state + transient response to the storm event) calculated along x in section 6.4.

In [ ]:
modelrun = "Run 52 max heads storm"
h_max = h_total.max(axis=1)

df_heads = pd.DataFrame(h_max.T, index=x)
df_heads.index.name = f"x [{units['x']}]"

f, ax = plt.subplots(figsize=(10, 5))
df_heads.plot(ylabel=f"Head [{units['head']}]", grid=True, ax=ax)
ax.set_title("Maximum heads per modellayer during a storm", loc="right")
tlt.finalize_and_save(f, savefolder, location_name, modelrun=modelrun, tag="maxheads", file_label=transect_name)

### 7.1 Total head over time at the piezometer locations

In [ ]:
# Select the x-locations to plot heads for:
xlocs = obslocs["x_crosssection"].unique()
# xlocs = np.array([0, 10, 100])

modelrun = "Run 61 storm event total head"

# Total head (steady + transient) [m NAP] over time, all layers
t = df_storm_event_offset.index.to_numpy()
h_t = tlt.calculate_head_t(mltr, xlocs, t, model_steady=mlss)

f, axes = tlt.plot_head_t(h_t, river=df_storm_event_offset["h_river"], units=units)
tlt.finalize_and_save(f, savefolder, location_name, modelrun=modelrun, tag="headTriv", file_label=transect_name)

## 8. Summary
All response factors in one table (one row per piezometer) and one Excel file:
- steady state (`rf_steady_mod`, `rf_steady_obs`)
- tidal: median over all tidal peaks (`rf_tidal_mod`, `rf_tidal_obs`) and the time lag in hours
- design high water of the storm event (MHW, `rf_mhw_mod`)

Both files in the 'results_final' folder of the transect are combined over all transects in notebook 60.

In [ ]:
# Model summary of this transect (one row), combined over all transects in notebook 60
model_summary = pd.DataFrame([{
    "transect": transect_name,
    "parameter_tab": mp_calibrated.attrs.get("tab_name"),
    "rmse_m": rmse,
    "leakage_factor_riverside_m": leakagefactor_riverside,
    "leakage_factor_landside_m": leakagefactor_landside,
    "flow_resistance_W1_d_per_m": flow_resistance_foreland,
    "flow_resistance_W3_d_per_m": flow_resistance_hinterland,
    "leakage_length_foreland_steady_m": fit_steady["leakage_length_foreland"],
    "leakage_length_hinterland_steady_m": fit_steady["leakage_length_hinterland"],
    "r_squared_steady": fit_steady["r_squared"],
    "leakage_length_foreland_mhw_m": fit_mhw["leakage_length_foreland"],
    "leakage_length_hinterland_mhw_m": fit_mhw["leakage_length_hinterland"],
    "r_squared_mhw": fit_mhw["r_squared"],
}]).set_index("transect")
tlt.export_to_excel(model_summary, savefolder_final / "model_summary.xlsx")

model_summary.T

In [ ]:
df_rf = tlt.combine_rf(obslocs, df_rf_steady, df_rf_tidal, df_rf_mhw)

rf_columns = [c for c in df_rf.columns if c.startswith("rf_")]
lag_columns = [c for c in df_rf.columns if c.startswith("lag_")]
tlt.export_to_excel(
    df_rf,
    savefolder_final / "responsefactors.xlsx",
    number_formats={**{c: "0.0%" for c in rf_columns}, **{c: "0.0" for c in lag_columns}},
)

df_rf.style.format({c: "{:.1%}" for c in rf_columns}, precision=2, na_rep="-")

Response factors along the cross-section, with the fitted leakage lengths (steady and MHW).

In [ ]:
f, axes = tlt.plot_rf_x(df_rf, fits=[fit_steady, fit_mhw], units=units)
tlt.finalize_and_save(f, savefolder_final, location_name, modelrun="Summary", tag="responsefactors_x", file_label=transect_name)

---

<!-- navigation -->
[← 51 Prepare model inputs](51%20Prepare%20model%20inputs.ipynb) | [Overview](README.md) | [60 Summary over transects →](60%20Summary%20over%20transects.ipynb)